# RetinaX Notebook 02: Modeling Progression, Explainability & Uncertainty Quantification

> **Clinical Disclaimer:** RetinaX is an academic research framework and is not clinically validated.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.models.custom_cnn import CustomFundusCNN
from src.models.classifier import build_model
from src.data.manifests import load_manifest
from src.experiments.ablation import run_supervised_experiment

cfg = load_config(preset_path=str(ROOT / "configs/lightweight_cpu.yaml"))
cnn = CustomFundusCNN(num_classes=5, num_blocks=3, base_channels=16)
print("CustomFundusCNN Receptive Field Table:", cnn.describe_receptive_field())
print("Parameter count:", sum(p.numel() for p in cnn.parameters()))

In [ ]:
manifest_df = load_manifest("synthetic_primary")
try:
    ext_df = load_manifest("synthetic_external")
except FileNotFoundError:
    ext_df = None

res = run_supervised_experiment(
    manifest_df=manifest_df,
    model_name="custom_cnn",
    run_id="notebook_demo_cnn",
    experiment_group="Notebook_Walkthrough",
    cfg=cfg,
    external_manifest_df=ext_df,
    epochs=3,
    image_size=128,
    seed=42,
)
print("Test QWK:", res["metrics"]["test"]["quadratic_weighted_kappa"])
print("ECE Before -> After Temp Scaling:", res["metrics"]["calibration_before"]["ece"], "->", res["metrics"]["calibration_after"]["ece"])
print("Conformal APS Empirical Coverage:", res["metrics"]["conformal_test"]["empirical_coverage"])

In [ ]:
from app.components.state import run_full_clinical_research_inference
from src.data.synthetic import render_synthetic_fundus_phantom
import matplotlib.pyplot as plt

phantom = render_synthetic_fundus_phantom(grade=3, seed=7, size=128)
wf = run_full_clinical_research_inference(
    phantom,
    checkpoint_path=res["checkpoint_path"],
    mc_passes=6,
    xai_method="gradcam",
)
print("Prediction:", wf["predicted_label"], "| Abstain:", wf["selective"]["status"], "| FOV Attribution:", wf["fov_attribution_ratio"])
plt.imshow(wf["overlay_rgb"])
plt.title(f"Grad-CAM Overlay ({wf['predicted_label']})")
plt.axis("off")
plt.show()